# Production Forecast & Inventory Decision Pipeline

In [1]:
import pandas as pd
import numpy as np

train = pd.read_pickle("../data/processed/train.pkl")
validation = pd.read_pickle("../data/processed/validation.pkl")
test = pd.read_pickle("../data/processed/test.pkl")

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (473436, 15)
Validation: (118664, 15)
Test: (121160, 15)


## Reconstruct Full Historical Demand

In [2]:
import pandas as pd
import numpy as np

# Load raw transaction data
df = pd.read_excel("../data/raw/online_retail.xlsx")

# Remove exact duplicate transactions
df_clean = df.drop_duplicates().copy()

# Identify cancellations
df_clean["is_cancellation"] = df_clean["InvoiceNo"].astype(str).str.startswith("C")

# Identify bad-debt adjustments
df_clean["is_bad_debt"] = df_clean["StockCode"].astype(str).str.upper().eq("B")

# Identify non-product transaction records
non_product_codes = {"D", "M", "POST", "DOT"}

df_clean["is_non_product"] = (
    df_clean["StockCode"].astype(str).str.upper().isin(non_product_codes)
)

# Identify damaged / unsaleable inventory adjustments
df_clean["is_damage"] = (
    df_clean["Description"]
    .fillna("")
    .astype(str)
    .str.contains(r"damaged|damage|unsaleable|destroyed", case=False, regex=True)
)

# Define genuine customer demand
df_clean["is_valid_demand"] = (
    (~df_clean["is_cancellation"])
    & (~df_clean["is_bad_debt"])
    & (~df_clean["is_non_product"])
    & (~df_clean["is_damage"])
    & ~((df_clean["Quantity"] < 0) & (df_clean["UnitPrice"] == 0))
)

# Keep genuine demand
df_demand = df_clean[df_clean["is_valid_demand"]].copy()

# Extract calendar date
df_demand["Date"] = df_demand["InvoiceDate"].dt.normalize()

# Aggregate to SKU × Day
daily_demand = (
    df_demand.groupby(["Date", "StockCode"], as_index=False)["Quantity"]
    .sum()
    .rename(columns={"Quantity": "Demand"})
)

print("Demand rows:", len(daily_demand))
print("Unique SKUs:", daily_demand["StockCode"].nunique())
print("Date range:", daily_demand["Date"].min(), "to", daily_demand["Date"].max())

Demand rows: 276167
Unique SKUs: 3936
Date range: 2010-12-01 00:00:00 to 2011-12-09 00:00:00


## Build Complete SKU Demand Calendar

In [3]:
# Determine each SKU's active period
sku_activity = (
    daily_demand.groupby("StockCode")["Date"]
    .agg(first_date="min", last_date="max")
    .reset_index()
)

# Create a continuous daily calendar for each SKU
sku_calendar = (
    sku_activity.set_index("StockCode")
    .apply(
        lambda row: pd.date_range(row["first_date"], row["last_date"], freq="D"), axis=1
    )
    .explode()
    .reset_index()
    .rename(columns={0: "Date"})
)

# Merge actual demand onto the calendar
demand_daily = sku_calendar.merge(daily_demand, on=["StockCode", "Date"], how="left")

# Missing SKU-days represent zero recorded demand
demand_daily["Demand"] = demand_daily["Demand"].fillna(0).astype(int)

# Sort chronologically
demand_daily = demand_daily.sort_values(["StockCode", "Date"]).reset_index(drop=True)

print("Rows:", len(demand_daily))
print("SKUs:", demand_daily["StockCode"].nunique())
print("Zero-demand rows:", (demand_daily["Demand"] == 0).sum())
print("Positive-demand rows:", (demand_daily["Demand"] > 0).sum())

Rows: 1076751
SKUs: 3936
Zero-demand rows: 800584
Positive-demand rows: 276167


## Save Processed Demand History

In [5]:
demand_daily.to_pickle("../data/processed/demand_daily.pkl")

print("Saved:", demand_daily.shape)

Saved: (1076751, 3)


## Prepare Features for Production Forecasting

In [4]:
# Keep only SKUs eligible for forecasting
sku_activity = (
    demand_daily.groupby("StockCode")["Demand"]
    .agg(active_days=lambda x: (x > 0).sum())
    .reset_index()
)

eligible_skus = sku_activity.loc[sku_activity["active_days"] >= 30, "StockCode"]

production_data = demand_daily[demand_daily["StockCode"].isin(eligible_skus)].copy()

production_data = production_data.sort_values(["StockCode", "Date"]).reset_index(
    drop=True
)

print("Eligible SKUs:", production_data["StockCode"].nunique())
print("Rows:", len(production_data))
print("Date range:", production_data["Date"].min(), "to", production_data["Date"].max())

Eligible SKUs: 2435
Rows: 783875
Date range: 2010-12-01 00:00:00 to 2011-12-09 00:00:00


## Create Forecasting Features

In [5]:
# Lag features
for lag in [1, 7, 14, 28]:
    production_data[f"lag_{lag}"] = production_data.groupby("StockCode")[
        "Demand"
    ].shift(lag)


# Rolling features
for window in [7, 28]:
    production_data[f"rolling_mean_{window}"] = production_data.groupby("StockCode")[
        "Demand"
    ].transform(lambda x: x.shift(1).rolling(window).mean())

production_data["rolling_std_7"] = production_data.groupby("StockCode")[
    "Demand"
].transform(lambda x: x.shift(1).rolling(7).std())


# Calendar features
production_data["day_of_week"] = production_data["Date"].dt.dayofweek

production_data["month"] = production_data["Date"].dt.month

production_data["week_of_year"] = (
    production_data["Date"].dt.isocalendar().week.astype(int)
)

production_data["is_weekend"] = (production_data["day_of_week"] >= 5).astype(int)


# Forecast feature list
model_features = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_std_7",
    "rolling_mean_28",
    "day_of_week",
    "month",
    "week_of_year",
    "is_weekend",
]

print("Features created:", len(model_features))

Features created: 11


## Train Final Production Model

In [27]:
from xgboost import XGBRegressor

# Create next-day demand target
production_data["target"] = production_data.groupby("StockCode")["Demand"].shift(-1)

# Keep rows where all forecasting features and target are available
production_train = production_data.dropna(subset=model_features + ["target"]).copy()

X_production = production_train[model_features]
y_production = production_train["target"]

print("Training rows:", len(production_train))
print("Features:", X_production.shape[1])

final_xgb = XGBRegressor(
    n_estimators=200,
    learning_rate=0.03,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

final_xgb.fit(X_production, y_production)

print("Final production model trained.")

Training rows: 713260
Features: 11
Final production model trained.


In [28]:
forecast_history = production_data[["StockCode", "Date", "Demand"]].copy()

forecast_history = forecast_history.sort_values(["StockCode", "Date"]).reset_index(
    drop=True
)

forecast_start = forecast_history["Date"].max() + pd.Timedelta(days=1)

forecast_dates = pd.date_range(start=forecast_start, periods=7, freq="D")

print("Forecast start:", forecast_start)
print("Forecast end:", forecast_dates[-1])

Forecast start: 2011-12-10 00:00:00
Forecast end: 2011-12-16 00:00:00


## **Don't run this section run the optimized one**

In [11]:
forecast_results = []

for forecast_date in forecast_dates:

    current_rows = []

    for sku in eligible_skus:

        sku_history = forecast_history[
            forecast_history["StockCode"] == sku
        ].sort_values("Date")

        demand_series = sku_history["Demand"]

        # Need at least 28 previous observations
        if len(demand_series) < 28:
            continue

        row = {
            "StockCode": sku,
            "Date": forecast_date,
            "lag_1": demand_series.iloc[-1],
            "lag_7": demand_series.iloc[-7],
            "lag_14": demand_series.iloc[-14],
            "lag_28": demand_series.iloc[-28],
            "rolling_mean_7": demand_series.iloc[-7:].mean(),
            "rolling_std_7": demand_series.iloc[-7:].std(),
            "rolling_mean_28": demand_series.iloc[-28:].mean(),
            "day_of_week": forecast_date.dayofweek,
            "month": forecast_date.month,
            "week_of_year": forecast_date.isocalendar().week,
            "is_weekend": int(forecast_date.dayofweek >= 5),
        }

        current_rows.append(row)

    current_features = pd.DataFrame(current_rows)

    predictions = final_xgb.predict(current_features[model_features])

    # Demand cannot be negative
    predictions = np.maximum(predictions, 0)

    current_features["Predicted_Demand"] = predictions

    forecast_results.append(current_features[["StockCode", "Date", "Predicted_Demand"]])

    # Add today's predictions to history
    new_history = current_features[["StockCode", "Date", "Predicted_Demand"]].rename(
        columns={"Predicted_Demand": "Demand"}
    )

    forecast_history = pd.concat([forecast_history, new_history], ignore_index=True)

    forecast_history = forecast_history.sort_values(["StockCode", "Date"]).reset_index(
        drop=True
    )


future_forecast = pd.concat(forecast_results, ignore_index=True)

print("Forecast rows:", len(future_forecast))
print(
    "Forecast range:",
    future_forecast["Date"].min(),
    "to",
    future_forecast["Date"].max(),
)

Forecast rows: 17045
Forecast range: 2011-12-10 00:00:00 to 2011-12-16 00:00:00


## Optimized Recursive Forecast Generation

In [29]:
# --------------------------------------------------
# Optimized Recursive 7-Day Forecast
# --------------------------------------------------

# Keep latest 28 days for every SKU
forecast_history = (
    production_data[["StockCode", "Date", "Demand"]]
    .sort_values(["StockCode", "Date"])
    .groupby("StockCode")
    .tail(28)
    .copy()
)

forecast_history = forecast_history.sort_values(["StockCode", "Date"]).reset_index(
    drop=True
)


# Forecast dates
forecast_start = demand_daily["Date"].max() + pd.Timedelta(days=1)

forecast_dates = pd.date_range(start=forecast_start, periods=7, freq="D")

forecast_results = []


# --------------------------------------------------
# Recursive forecasting
# --------------------------------------------------

for forecast_date in forecast_dates:

    # Pivot the latest history into rows = SKU
    history_pivot = (
        forecast_history.sort_values(["StockCode", "Date"])
        .groupby("StockCode")["Demand"]
        .apply(list)
    )

    feature_data = pd.DataFrame(
        {
            "StockCode": history_pivot.index,
            "lag_1": history_pivot.apply(lambda x: x[-1]),
            "lag_7": history_pivot.apply(lambda x: x[-7]),
            "lag_14": history_pivot.apply(lambda x: x[-14]),
            "lag_28": history_pivot.apply(lambda x: x[-28]),
            "rolling_mean_7": history_pivot.apply(lambda x: np.mean(x[-7:])),
            "rolling_std_7": history_pivot.apply(lambda x: np.std(x[-7:], ddof=1)),
            "rolling_mean_28": history_pivot.apply(lambda x: np.mean(x[-28:])),
        }
    )

    # Calendar features
    feature_data["day_of_week"] = forecast_date.weekday()

    feature_data["month"] = forecast_date.month

    feature_data["week_of_year"] = forecast_date.isocalendar().week

    feature_data["is_weekend"] = int(forecast_date.weekday() >= 5)

    # Ensure feature order matches training
    predictions = final_xgb.predict(feature_data[model_features])

    # Demand cannot be negative
    predictions = np.maximum(predictions, 0)

    # Store predictions
    current_forecast = feature_data[["StockCode"]].copy()

    current_forecast["Date"] = forecast_date

    current_forecast["Predicted_Demand"] = predictions

    forecast_results.append(current_forecast)

    # Add predictions to history
    new_history = current_forecast[["StockCode", "Date", "Predicted_Demand"]].rename(
        columns={"Predicted_Demand": "Demand"}
    )

    forecast_history = pd.concat([forecast_history, new_history], ignore_index=True)

    # Keep latest 28 observations
    forecast_history = (
        forecast_history.sort_values(["StockCode", "Date"])
        .groupby("StockCode")
        .tail(28)
        .reset_index(drop=True)
    )


# --------------------------------------------------
# Combine forecasts
# --------------------------------------------------

future_forecast = pd.concat(forecast_results, ignore_index=True)


# --------------------------------------------------
# Validation
# --------------------------------------------------

print("Forecast rows:", len(future_forecast))

print(
    "Forecast range:",
    future_forecast["Date"].min(),
    "to",
    future_forecast["Date"].max(),
)

print("Forecast SKUs:", future_forecast["StockCode"].nunique())

Forecast rows: 17045
Forecast range: 2011-12-10 00:00:00 to 2011-12-16 00:00:00
Forecast SKUs: 2435


In [30]:
future_forecast.head(10)

,StockCode,Date,Predicted_Demand
0,10002,2011-12-10,7.487536
1,10125,2011-12-10,3.322407
2,10133,2011-12-10,22.588724
3,10135,2011-12-10,4.938139
4,11001,2011-12-10,6.178058
5,15034,2011-12-10,4.080483
6,15036,2011-12-10,7.643989
7,15039,2011-12-10,5.094263
8,16008,2011-12-10,16.803812
9,16011,2011-12-10,7.978553


In [31]:
future_forecast["Predicted_Demand"].describe()

count    17045.000000
mean        11.522584
std         20.303448
min          0.000000
25%          2.363475
50%          5.131384
75%         12.666938
max        392.584290
Name: Predicted_Demand, dtype: float64

In [32]:
future_forecast.groupby("Date")["Predicted_Demand"].agg(["count", "mean", "sum"])

,count,mean,sum
Date,,,
2011-12-10,2435,6.982355,17002.035156
2011-12-11,2435,11.419450,27806.361328
2011-12-12,2435,15.323017,37311.546875
2011-12-13,2435,15.472939,37676.605469
2011-12-14,2435,15.848455,38590.988281
2011-12-15,2435,12.149132,29583.134766
2011-12-16,2435,3.462741,8431.774414


## Build Forecast-Based Inventory Decisions

In [33]:
# Aggregate 7-day forecast by SKU
sku_forecast = (
    future_forecast.groupby("StockCode")
    .agg(
        forecast_7_day_demand=("Predicted_Demand", "sum"),
        avg_forecast_daily_demand=("Predicted_Demand", "mean"),
    )
    .reset_index()
)

print("SKUs:", sku_forecast["StockCode"].nunique())
print("Rows:", len(sku_forecast))

SKUs: 2435
Rows: 2435


In [34]:
# Historical demand statistics for inventory planning
inventory_stats = (
    production_data.groupby("StockCode")["Demand"]
    .agg(mean_daily_demand="mean", std_daily_demand="std")
    .reset_index()
)

# Inventory planning assumptions
lead_time = 7
service_level = 0.95
z_score = 1.645

# Safety stock
inventory_stats["safety_stock"] = (
    z_score * inventory_stats["std_daily_demand"] * np.sqrt(lead_time)
)

# Expected demand during lead time
inventory_stats["lead_time_demand"] = inventory_stats["mean_daily_demand"] * lead_time

# Historical reorder point
inventory_stats["historical_reorder_point"] = (
    inventory_stats["lead_time_demand"] + inventory_stats["safety_stock"]
)

print("Inventory statistics created.")

Inventory statistics created.


In [35]:
# Combine forecast with inventory statistics
sku_inventory = inventory_stats.merge(sku_forecast, on="StockCode", how="inner")

# Forecast-based reorder point
sku_inventory["forecast_reorder_point"] = (
    sku_inventory["forecast_7_day_demand"] + sku_inventory["safety_stock"]
)

print("Decision rows:", len(sku_inventory))
print("Missing values:", sku_inventory.isna().sum().sum())

Decision rows: 2435
Missing values: 0


In [36]:
sku_inventory[
    ["StockCode", "forecast_7_day_demand", "safety_stock", "forecast_reorder_point"]
].head(10)

,StockCode,forecast_7_day_demand,safety_stock,forecast_reorder_point
0,10002,104.114395,103.259961,207.374357
1,10125,33.502556,61.104338,94.606894
2,10133,169.680191,94.317646,263.997837
3,10135,53.549938,86.659088,140.209026
4,11001,42.697689,95.257843,137.955532
5,15034,42.251793,426.915771,469.167564
6,15036,107.783401,833.824491,941.607893
7,15039,48.828217,106.700674,155.528891
8,16008,206.875824,145.008258,351.884082
9,16011,82.288841,78.135034,160.423875


In [37]:
# Calculate simulated current inventory
recent_inventory = (
    demand_daily[demand_daily["StockCode"].isin(eligible_skus)]
    .sort_values(["StockCode", "Date"])
    .groupby("StockCode")
    .tail(14)
    .groupby("StockCode")["Demand"]
    .sum()
    .reset_index(name="simulated_inventory")
)

print("SKUs:", recent_inventory["StockCode"].nunique())
print("Rows:", len(recent_inventory))
print("Missing inventory values:", recent_inventory["simulated_inventory"].isna().sum())

SKUs: 2435
Rows: 2435
Missing inventory values: 0


In [38]:
sku_inventory = sku_inventory.merge(recent_inventory, on="StockCode", how="inner")

print("Final decision rows:", len(sku_inventory))

Final decision rows: 2435


In [39]:
# Calculate inventory gap
sku_inventory["inventory_gap"] = (
    sku_inventory["forecast_reorder_point"] - sku_inventory["simulated_inventory"]
)

# Percentage gap relative to reorder point
sku_inventory["inventory_gap_pct"] = (
    sku_inventory["inventory_gap"] / sku_inventory["forecast_reorder_point"]
) * 100

# Assign risk levels
sku_inventory["risk_level"] = pd.cut(
    sku_inventory["inventory_gap_pct"],
    bins=[-np.inf, 0, 25, 50, np.inf],
    labels=["Healthy", "Watch", "High", "Critical"],
)

print(
    sku_inventory["risk_level"]
    .value_counts()
    .reindex(["Healthy", "Watch", "High", "Critical"], fill_value=0)
)

risk_level
Healthy     574
Watch       424
High        494
Critical    943
Name: count, dtype: int64


In [40]:
sku_inventory[
    [
        "StockCode",
        "simulated_inventory",
        "forecast_reorder_point",
        "inventory_gap",
        "inventory_gap_pct",
        "risk_level",
    ]
].head(10)

,StockCode,simulated_inventory,forecast_reorder_point,inventory_gap,inventory_gap_pct,risk_level
0,10002,63,207.374357,144.374357,69.620159,Critical
1,10125,26,94.606894,68.606894,72.517859,Critical
2,10133,298,263.997837,-34.002163,-12.879713,Healthy
3,10135,117,140.209026,23.209026,16.553161,Watch
4,11001,76,137.955532,61.955532,44.909784,High
5,15034,18,469.167564,451.167564,96.163418,Critical
6,15036,157,941.607893,784.607893,83.326393,Critical
7,15039,48,155.528891,107.528891,69.137567,Critical
8,16008,624,351.884082,-272.115918,-77.331125,Healthy
9,16011,73,160.423875,87.423875,54.495551,Critical


In [41]:
def recommend_action(risk_level):
    if risk_level == "Critical":
        return "Reorder immediately"
    elif risk_level == "High":
        return "Reorder soon"
    elif risk_level == "Watch":
        return "Monitor closely"
    else:
        return "No action"


sku_inventory["recommended_action"] = sku_inventory["risk_level"].map(recommend_action)

print(
    sku_inventory["recommended_action"]
    .value_counts()
    .reindex(
        ["Reorder immediately", "Reorder soon", "Monitor closely", "No action"],
        fill_value=0,
    )
)

recommended_action
Reorder immediately    943
Reorder soon           494
Monitor closely        424
No action              574
Name: count, dtype: int64


In [42]:
decision_columns = [
    "StockCode",
    "mean_daily_demand",
    "std_daily_demand",
    "safety_stock",
    "forecast_7_day_demand",
    "avg_forecast_daily_demand",
    "forecast_reorder_point",
    "simulated_inventory",
    "inventory_gap",
    "inventory_gap_pct",
    "risk_level",
    "recommended_action",
]

decision_data = sku_inventory[decision_columns].copy()

decision_data.to_csv("../data/processed/stock_inventory_decisions.csv", index=False)

print("Saved:", decision_data.shape)
print("Missing values:", decision_data.isna().sum().sum())

Saved: (2435, 12)
Missing values: 0


In [43]:
decision_data.head()

,StockCode,mean_daily_demand,std_daily_demand,safety_stock,forecast_7_day_demand,avg_forecast_daily_demand,forecast_reorder_point,simulated_inventory,inventory_gap,inventory_gap_pct,risk_level,recommended_action
0,10002,7.482014,23.725591,103.259961,104.114395,14.873485,207.374357,63,144.374357,69.620159,Critical,Reorder immediately
1,10125,3.462567,14.039677,61.104338,33.502556,4.786079,94.606894,26,68.606894,72.517859,Critical,Reorder immediately
2,10133,10.163701,21.670954,94.317646,169.680191,24.240026,263.997837,298,-34.002163,-12.879713,Healthy,No action
3,10135,5.975871,19.911281,86.659088,53.549938,7.649991,140.209026,117,23.209026,16.553161,Watch,Monitor closely
4,11001,4.353100,21.886979,95.257843,42.697689,6.099670,137.955532,76,61.955532,44.909784,High,Reorder soon


In [44]:
print("Training target:", production_train["target"].name)
print("Training rows:", len(production_train))
print("Target mean:", production_train["target"].mean())
print("Target min:", production_train["target"].min())
print("Target max:", production_train["target"].max())

Training target: target
Training rows: 713260
Target mean: 6.64346802007683
Target min: 0.0
Target max: 4848.0


In [45]:
print("Decision rows:", len(decision_data))
print("Forecast rows:", len(future_forecast))
print("Missing decision values:", decision_data.isna().sum().sum())
print("Missing forecast values:", future_forecast.isna().sum().sum())

Decision rows: 2435
Forecast rows: 17045
Missing decision values: 0
Missing forecast values: 0


## Save Production Model

In [46]:
import joblib

joblib.dump(final_xgb, "../models/final_xgb_model.pkl")

print("Model saved successfully.")

Model saved successfully.


In [47]:
import os

print("Model exists:", os.path.exists("../models/final_xgb_model.pkl"))

Model exists: True


In [48]:
future_forecast.to_csv("../data/processed/future_7_day_forecast.csv", index=False)

print("Forecast saved:", future_forecast.shape)

Forecast saved: (17045, 3)


In [49]:
# Save demand history for Streamlit
production_data[["StockCode", "Date", "Demand"]].to_csv(
    "../data/processed/production_demand_history.csv", index=False
)

print("Demand history saved:", production_data[["StockCode", "Date", "Demand"]].shape)

Demand history saved: (783875, 3)
